# 05 - Analysis

Compute all metrics and generate figures.

No GPU needed for this notebook.

In [ ]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/13-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "-b", "13-4-2026",
        "https://github.com/JackHopkins/legibility.git",
        str(REPO_DIR)
    ], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 150, "font.size": 11})

## Load all results

In [ ]:
ALL_CONDITIONS = [
    "no_cot", "normal", "self_prefill", "cross_prefill",
    "paraphrase_self", "paraphrase_cross",
    "heavy_paraphrase_self", "heavy_paraphrase_cross",
    "shuffled_steps", "corrupted_numbers",
]

rows = []
for cond in ALL_CONDITIONS:
    for p in PREFILL_CACHE.glob(f"{cond}_*.json"):
        r = json.loads(p.read_text())
        rows.append({
            "condition": cond,
            "problem_id": r["problem_id"],
            "predicted": r["predicted_answer"],
            "gold": r["gold_answer"],
            "correct": r["predicted_answer"] == r["gold_answer"],
        })

df = pd.DataFrame(rows)
print(f"Total results: {len(df)}")
print(f"\nResults per condition:")
print(df.groupby("condition").size().reindex(ALL_CONDITIONS))

## 1. Main results table

In [ ]:
def bootstrap_ci(correct_arr, n_bootstrap=10000, ci=0.95, seed=42):
    """Compute bootstrap confidence interval for accuracy."""
    rng = np.random.RandomState(seed)
    n = len(correct_arr)
    boot_means = []
    for _ in range(n_bootstrap):
        sample = rng.choice(correct_arr, size=n, replace=True)
        boot_means.append(sample.mean())
    boot_means = np.sort(boot_means)
    lo = boot_means[int((1 - ci) / 2 * n_bootstrap)]
    hi = boot_means[int((1 + ci) / 2 * n_bootstrap)]
    return lo, hi


# Compute accuracy and CI per condition
acc_no_cot = df[df.condition == "no_cot"]["correct"].mean()
acc_self = df[df.condition == "self_prefill"]["correct"].mean()
total_cot_value = acc_self - acc_no_cot

results = []
for cond in ALL_CONDITIONS:
    subset = df[df.condition == cond]
    if len(subset) == 0:
        continue
    acc = subset["correct"].mean()
    lo, hi = bootstrap_ci(subset["correct"].values)
    L = (acc - acc_no_cot) / total_cot_value if total_cot_value > 0 else None
    results.append({
        "Condition": cond,
        "Accuracy": f"{acc:.3f}",
        "95% CI": f"[{lo:.3f}, {hi:.3f}]",
        "Legibility (L)": f"{L:.3f}" if L is not None else "N/A",
        "N": len(subset),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Save
results_df.to_csv(RESULTS_DIR / "main_results.csv", index=False)

## 2. Information decomposition

In [ ]:
# Compute all accuracy values
accs = {}
for cond in ALL_CONDITIONS:
    subset = df[df.condition == cond]
    if len(subset) > 0:
        accs[cond] = subset["correct"].mean()

print("Accuracy values:")
for k, v in accs.items():
    print(f"  {k:30s}: {v:.4f}")

# Decomposition
total_value = accs["self_prefill"] - accs["no_cot"]
semantic_content = accs.get("paraphrase_self", 0) - accs["no_cot"]
token_encoding = accs["self_prefill"] - accs.get("paraphrase_self", accs["self_prefill"])
universal_legibility = accs.get("paraphrase_cross", 0) - accs["no_cot"]
model_specific_semantic = accs.get("paraphrase_self", 0) - accs.get("paraphrase_cross", 0)
cross_transfer_premium = accs.get("cross_prefill", 0) - accs.get("paraphrase_cross", 0)
structural_info = accs.get("paraphrase_self", 0) - accs.get("heavy_paraphrase_self", 0)

L = universal_legibility / total_value if total_value > 0 else None

print(f"\n--- Information Decomposition ---")
print(f"Total COT value:          {total_value:.4f}")
print(f"Semantic content:         {semantic_content:.4f} ({semantic_content/total_value:.1%} of total)")
print(f"Token encoding:           {token_encoding:.4f} ({token_encoding/total_value:.1%} of total)")
print(f"Universal legibility:     {universal_legibility:.4f} ({universal_legibility/total_value:.1%} of total)")
print(f"Model-specific semantic:  {model_specific_semantic:.4f}")
print(f"Cross-model transfer premium: {cross_transfer_premium:.4f}")
print(f"Structural information:   {structural_info:.4f}")
print(f"\nLEGIBILITY SCORE (L):     {L:.4f}" if L else "\nLEGIBILITY SCORE: N/A")

## 3. Ablation consistency check

In [ ]:
diff = abs(accs.get("self_prefill", 0) - accs.get("normal", 0))
print(f"self_prefill accuracy: {accs.get('self_prefill', 0):.4f}")
print(f"normal accuracy:       {accs.get('normal', 0):.4f}")
print(f"Difference:            {diff:.4f}")
if diff > 0.02:
    print("WARNING: self_prefill differs from normal by >2pp. Two-pass setup may introduce artifacts.")
else:
    print("OK: self_prefill ~ normal (within 2pp). Methodology validated.")

## 4. Figure: 2x2 Heatmap

In [ ]:
# 2x2 heatmap: self/cross x original/paraphrased
heatmap_data = np.array([
    [accs.get("self_prefill", 0), accs.get("cross_prefill", 0)],
    [accs.get("paraphrase_self", 0), accs.get("paraphrase_cross", 0)],
])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".3f", cmap="YlOrRd_r",
    xticklabels=["Same model\n(Qwen3-4B)", "Different model\n(Gemma-3-4B)"],
    yticklabels=["Original text", "Paraphrased text"],
    vmin=accs.get("no_cot", 0), vmax=accs.get("self_prefill", 1),
    ax=ax,
    linewidths=2,
)
ax.set_title("COT Legibility: Semantic Bottleneck 2x2", fontsize=13, fontweight="bold")
ax.set_xlabel("Reader model")
ax.set_ylabel("COT text")

# Annotate deltas
no_cot_acc = accs.get("no_cot", 0)
ax.text(0.5, -0.15, f"No-COT baseline: {no_cot_acc:.3f}",
        ha="center", transform=ax.transAxes, fontsize=10, style="italic")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "heatmap_2x2.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Figure: Information Decomposition Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Stacked bar components
baseline = accs.get("no_cot", 0)
universal = universal_legibility
model_specific = model_specific_semantic
token_enc = token_encoding

# Error bars via bootstrap
def get_acc_ci(cond):
    subset = df[df.condition == cond]["correct"].values
    if len(subset) == 0:
        return 0, 0, 0
    acc = subset.mean()
    lo, hi = bootstrap_ci(subset)
    return acc, lo, hi

conditions_to_plot = [
    "no_cot", "paraphrase_cross", "paraphrase_self", "self_prefill", "normal"
]
labels = [
    "No COT", "Paraphrase\nCross", "Paraphrase\nSelf", "Self\nPrefill", "Normal"
]
colors = ["#95a5a6", "#27ae60", "#2980b9", "#8e44ad", "#e74c3c"]

acc_vals = []
ci_errs = []
for cond in conditions_to_plot:
    acc, lo, hi = get_acc_ci(cond)
    acc_vals.append(acc)
    ci_errs.append([acc - lo, hi - acc])

ci_errs = np.array(ci_errs).T

bars = ax.bar(range(len(conditions_to_plot)), acc_vals,
              yerr=ci_errs, capsize=4, color=colors, edgecolor="black", linewidth=0.5)

for bar, val in zip(bars, acc_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)

ax.set_xticks(range(len(conditions_to_plot)))
ax.set_xticklabels(labels)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy by Condition (with 95% CI)", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "accuracy_by_condition.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Figure: Full Condition Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

all_acc = []
all_ci = []
for cond in ALL_CONDITIONS:
    acc, lo, hi = get_acc_ci(cond)
    all_acc.append(acc)
    all_ci.append([acc - lo, hi - acc])

all_ci = np.array(all_ci).T

bar_colors = [
    "#95a5a6",  # no_cot
    "#e74c3c",  # normal
    "#8e44ad",  # self_prefill
    "#9b59b6",  # cross_prefill
    "#2980b9",  # paraphrase_self
    "#27ae60",  # paraphrase_cross
    "#3498db",  # heavy_paraphrase_self
    "#2ecc71",  # heavy_paraphrase_cross
    "#f39c12",  # shuffled_steps
    "#e67e22",  # corrupted_numbers
]

bars = ax.bar(range(len(ALL_CONDITIONS)), all_acc,
              yerr=all_ci, capsize=3, color=bar_colors,
              edgecolor="black", linewidth=0.5)

for bar, val in zip(bars, all_acc):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=8, rotation=45)

ax.set_xticks(range(len(ALL_CONDITIONS)))
ax.set_xticklabels([c.replace("_", "\n") for c in ALL_CONDITIONS], fontsize=8)
ax.set_ylabel("Accuracy")
ax.set_title("All Conditions: Accuracy with 95% CI", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1)
ax.axhline(y=accs.get("no_cot", 0), color="gray", linestyle="--", alpha=0.5, label="No-COT baseline")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "all_conditions.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Figure: Information Decomposition Stacked Bar

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# Components of the COT value decomposition
components = [
    ("No-COT baseline", baseline, "#bdc3c7"),
    ("Universally legible", max(0, universal), "#27ae60"),
    ("Model-specific semantic", max(0, model_specific), "#f39c12"),
    ("Token-level encoding", max(0, token_enc), "#e74c3c"),
]

bottom = 0
for label, val, color in components:
    ax.bar(0, val, bottom=bottom, color=color, edgecolor="black",
           linewidth=0.5, label=f"{label} ({val:.3f})", width=0.5)
    if val > 0.02:
        ax.text(0, bottom + val / 2, f"{val:.3f}", ha="center", va="center",
                fontsize=10, fontweight="bold")
    bottom += val

ax.set_xlim(-0.5, 0.5)
ax.set_ylim(0, 1)
ax.set_xticks([0])
ax.set_xticklabels(["COT Value\nDecomposition"])
ax.set_ylabel("Accuracy")
ax.set_title("Information Decomposition", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "decomposition.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Per-problem failure analysis

In [ ]:
# Find problems where self_prefill succeeds but paraphrase_cross fails
self_results = df[df.condition == "self_prefill"].set_index("problem_id")
cross_para_results = df[df.condition == "paraphrase_cross"].set_index("problem_id")

common_ids = self_results.index.intersection(cross_para_results.index)

failures = []
for pid in common_ids:
    if self_results.loc[pid, "correct"] and not cross_para_results.loc[pid, "correct"]:
        failures.append(pid)

print(f"Problems where self_prefill correct but paraphrase_cross wrong: {len(failures)}/{len(common_ids)}")
print(f"Failure rate: {len(failures)/len(common_ids):.1%}")

# Show a few failure examples
for pid in failures[:5]:
    cot_file = COT_CACHE / f"{pid}.json"
    if cot_file.exists():
        cot = json.loads(cot_file.read_text())
        print(f"\n--- Problem {pid} ---")
        print(f"Question: {cot['question'][:200]}...")
        print(f"Gold: {cot['gold_answer']}")
        print(f"COT (first 200 chars): {cot['cot_text'][:200]}")

## 9. Difficulty analysis

In [ ]:
# Estimate difficulty by number of reasoning steps in the COT
step_counts = {}
for p in COT_CACHE.glob("*.json"):
    r = json.loads(p.read_text())
    pid = r["problem_id"]
    # Count non-empty lines as proxy for step count
    lines = [l for l in r["cot_text"].split("\n") if l.strip()]
    step_counts[pid] = len(lines)

# Merge with results
df["steps"] = df["problem_id"].map(step_counts)

# Bin by step count
df["step_bin"] = pd.cut(df["steps"], bins=[0, 3, 6, 10, 15, 100],
                         labels=["1-3", "4-6", "7-10", "11-15", "16+"])

# Compute legibility L per bin for paraphrase_cross
fig, ax = plt.subplots(figsize=(8, 5))

bins = df["step_bin"].dropna().unique()
bins = sorted(bins, key=str)

l_values = []
l_errors = []

for b in bins:
    bin_pids = df[(df.step_bin == b)]["problem_id"].unique()

    self_acc = df[(df.condition == "self_prefill") & (df.problem_id.isin(bin_pids))]["correct"].mean()
    no_cot_acc = df[(df.condition == "no_cot") & (df.problem_id.isin(bin_pids))]["correct"].mean()
    para_cross_acc = df[(df.condition == "paraphrase_cross") & (df.problem_id.isin(bin_pids))]["correct"].mean()

    cot_val = self_acc - no_cot_acc
    if cot_val > 0.01:
        L_bin = (para_cross_acc - no_cot_acc) / cot_val
    else:
        L_bin = None

    if L_bin is not None:
        l_values.append(L_bin)
        # Bootstrap CI for L in this bin
        self_arr = df[(df.condition == "self_prefill") & (df.problem_id.isin(bin_pids))]["correct"].values
        nc_arr = df[(df.condition == "no_cot") & (df.problem_id.isin(bin_pids))]["correct"].values
        pc_arr = df[(df.condition == "paraphrase_cross") & (df.problem_id.isin(bin_pids))]["correct"].values

        rng = np.random.RandomState(42)
        boot_L = []
        n_min = min(len(self_arr), len(nc_arr), len(pc_arr))
        for _ in range(5000):
            idx = rng.choice(n_min, size=n_min, replace=True)
            s_acc = self_arr[idx].mean() if len(self_arr) >= n_min else self_acc
            n_acc = nc_arr[idx].mean() if len(nc_arr) >= n_min else no_cot_acc
            p_acc = pc_arr[idx].mean() if len(pc_arr) >= n_min else para_cross_acc
            cv = s_acc - n_acc
            if cv > 0.01:
                boot_L.append((p_acc - n_acc) / cv)
        if boot_L:
            boot_L = np.sort(boot_L)
            lo = boot_L[int(0.025 * len(boot_L))]
            hi = boot_L[int(0.975 * len(boot_L))]
            l_errors.append([L_bin - lo, hi - L_bin])
        else:
            l_errors.append([0, 0])
    else:
        l_values.append(np.nan)
        l_errors.append([0, 0])

valid = [(b, v, e) for b, v, e in zip(bins, l_values, l_errors) if not np.isnan(v)]
if valid:
    vbins, vvals, verrs = zip(*valid)
    verrs = np.array(verrs).T
    ax.bar(range(len(vbins)), vvals, yerr=verrs, capsize=4,
           color="#27ae60", edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(vbins)))
    ax.set_xticklabels(vbins)

ax.set_xlabel("Number of COT steps")
ax.set_ylabel("Legibility score (L)")
ax.set_title("Legibility vs. Problem Difficulty", fontsize=13, fontweight="bold")
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.3)
ax.axhline(y=0.5, color="red", linestyle="--", alpha=0.3)
ax.set_ylim(0, 1.5)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "legibility_vs_difficulty.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Summary

In [ ]:
print("=" * 60)
print("COT LEGIBILITY EXPERIMENT - SUMMARY")
print("=" * 60)
print(f"\nDataset: GSM8K test ({len(df[df.condition == 'no_cot'])} problems)")
print(f"Primary model: {PRIMARY_MODEL}")
print(f"Cross model: {CROSS_MODEL}")
print(f"Paraphraser: {PARAPHRASER_MODEL}")
print(f"\n--- Key Results ---")
print(f"No-COT baseline:     {accs.get('no_cot', 0):.3f}")
print(f"Normal (COT):        {accs.get('normal', 0):.3f}")
print(f"Self-prefill:        {accs.get('self_prefill', 0):.3f}")
print(f"Cross-prefill:       {accs.get('cross_prefill', 0):.3f}")
print(f"Paraphrase self:     {accs.get('paraphrase_self', 0):.3f}")
print(f"Paraphrase cross:    {accs.get('paraphrase_cross', 0):.3f}")
print(f"\n--- Headline ---")
print(f"LEGIBILITY SCORE (L): {L:.3f}" if L else "LEGIBILITY SCORE: N/A")
if L is not None:
    if L > 0.8:
        print("INTERPRETATION: HIGH legibility. COT monitoring is viable.")
    elif L > 0.5:
        print("INTERPRETATION: MODERATE legibility. Monitoring captures some but not all COT value.")
    else:
        print("INTERPRETATION: LOW legibility. COT monitoring is unreliable.")
print("=" * 60)

# Save decomposition
decomposition = {
    "total_cot_value": total_value,
    "semantic_content": semantic_content,
    "token_encoding": token_encoding,
    "universal_legibility": universal_legibility,
    "model_specific_semantic": model_specific_semantic,
    "cross_transfer_premium": cross_transfer_premium,
    "structural_information": structural_info,
    "legibility_score_L": L,
    "accuracies": accs,
}
with open(RESULTS_DIR / "decomposition.json", "w") as f:
    json.dump(decomposition, f, indent=2, default=str)

print(f"\nResults saved to {RESULTS_DIR}")
print(f"Figures saved to {FIGURES_DIR}")